In [1]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('KO_weekly_return_volatility.csv')
daily_data = pd.read_csv('KO_weekly_return_detailed.csv')

train = df[df['Year'].isin([2021, 2022, 2023])]
test = df[df['Year'].isin([2024, 2025])]
daily_test = daily_data[daily_data['Year'].isin([2024, 2025])].reset_index(drop=True)

X_train = train[['mean_return', 'volatility']].values
Y_train = train['decision'].values
X_test = test[['mean_return', 'volatility']].values
Y_test = test['decision'].values

print(X_train.shape)

gnb = GaussianNB()
gnb.fit(X_train, Y_train)
predictions = gnb.predict(X_test)

(156, 2)


## Q1 : Implement Gaussian Naive Bayes and compute accuracy for testing period (years 4-5)

In [2]:
accuracy = round(gnb.score(X_test, Y_test) * 100, 2)
print(f'Test accuracy: {accuracy}%')

Test accuracy: 69.81%


## Q2 : Confusion matrix for testing period


In [3]:
cm = confusion_matrix(Y_test, predictions, labels=['YES', 'NO'])
print('Confusion Matrix:')
print(f'  Predicted YES  Predicted NO')
print(f'Actual YES  {cm[0][0]}           {cm[0][1]}')
print(f'Actual NO   {cm[1][0]}           {cm[1][1]}')

Confusion Matrix:
  Predicted YES  Predicted NO
Actual YES  1           32
Actual NO   0           73


## Q3 : True positive rate and true negative rate for testing period


In [4]:
TP, FN, FP, TN = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
print(f'TPR (sensitivity): {round(TP / (TP + FN) * 100, 2)}%')
print(f'TNR (specificity): {round(TN / (TN + FP) * 100, 2)}%')

TPR (sensitivity): 3.03%
TNR (specificity): 100.0%


## Q4 : Trading strategy vs buy-and-hold


In [5]:
test_reset = test.reset_index(drop=True).copy()
test_reset['nb_prediction'] = predictions

initial_balance = 100
nb_balance = initial_balance
position = None

for week_idx in range(len(test_reset)):
    current_week = test_reset.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                nb_balance = nb_balance * (1 + row['Return'] / 100)

    if week_idx < len(test_reset) - 1:
        next_prediction = test_reset.iloc[week_idx + 1]['nb_prediction']
        position = 'invested' if next_prediction == 'YES' else None

bh_balance = initial_balance
for idx, row in daily_test.iterrows():
    if pd.notna(row['Return']):
        bh_balance = bh_balance * (1 + row['Return'] / 100)

print(f'Naive Bayes Strategy Final Balance: ${nb_balance:.2f}')
print(f'Buy and Hold Final Balance: ${bh_balance:.2f}')
winner = 'Naive Bayes Strategy' if nb_balance > bh_balance else 'Buy-and-Hold'
print(f'{winner} results in a larger amount at the end of the testing period')

Naive Bayes Strategy Final Balance: $107.88
Buy and Hold Final Balance: $126.14
Buy-and-Hold results in a larger amount at the end of the testing period
